# 05 · Medallion
*Three vaults: raw as received (bronze), cleaned (silver), briefings for the Avengers (gold).*

Build silver and gold from the bronze tables of chapter 03. This is the simple full-refresh version, and it has a
**deliberate flaw**: no quality gate. Step 5 shows where the flaw ends up. Chapter 06 fixes it, chapter 08 makes it incremental.

| Concept | Where you meet it |
|---|---|
| Bronze → silver → gold layers, one schema each | the next section, then the whole notebook |
| Deduplicate to the latest record per key | Step 1 |
| Gold named after the question it answers | Steps 3 and 4 |
| Full refresh with `CREATE OR REPLACE` (idempotent) | Steps 1 to 4 |

Deep dive: [docs/05-medallion.md](../../docs/05-medallion.md)

**Needs:** chapter 03 ran completely (`shield.bronze.incident_reports` with all five files, `shield.bronze.locations`) and the schemas from chapter 01.
The **Expected** numbers assume the dataset from chapter 02 (10,000 incidents, seed 42, 5 % dirty); other data gives other numbers and a ❌, which is not a bug. The future-timestamp counts also depend on the day chapter 02 generated the data: those timestamps are 30 to 400 days ahead of that day, so some drift into the past as time goes by.

## The medallion architecture

**The idea:** data gets better in stages. You keep every stage as its own set of Delta tables, and each stage is built only from the one before it.
It is a design pattern, not a Databricks feature: nothing enforces it. The layers exist because of the names and rules your team keeps.

```
 files in the landing volume
        │  Auto Loader (chapter 03)
        ▼
 ┌──────────────┐   Steps 1-2    ┌──────────────┐   Steps 3-4    ┌──────────────┐
 │    BRONZE    │ ─────────────▶ │    SILVER    │ ─────────────▶ │     GOLD     │
 │ as received  │  type, dedupe  │ trustworthy  │   aggregate    │ one question │
 │ shield.bronze│                │ shield.silver│                │ per table    │
 └──────────────┘                └──────────────┘                └──────────────┘
                                                                  shield.gold
```

| Layer | Contract | Tables in this repo | Built by |
|---|---|---|---|
| **Bronze** | Raw data exactly as it arrived, plus lineage columns (`_source_file`, `_ingested_at`). Append-only. Duplicates and bad rows stay. | `incident_reports`, `locations`, `agents`, `threat_assessments` | chapter 03 |
| **Silver** | One typed row per business entity or event. This is the base everyone trusts. | `incident_reports`, `locations` | Steps 1 and 2 |
| **Gold** | Aggregates shaped for one use case, named after the question they answer. | `daily_threat_briefing`, `location_risk` | Steps 3 and 4 |

**How this notebook implements it**

* **One Unity Catalog schema per layer.** `shield.bronze`, `shield.silver` and `shield.gold` (created in chapter 01). The schema name tells you the layer, and the table keeps its name across layers (`incident_reports`), so lineage reads naturally.
* **Each layer reads only the layer before it.** Silver reads bronze, gold reads silver. Gold never touches bronze, so a fix in silver reaches every gold table.
* **Silver = transformation of the rows.** Step 1 parses the timestamp text into `reported_ts` and keeps the newest record per `incident_id` with `row_number()` over a window. Step 2 selects the columns the gold joins need.
* **Gold = SQL aggregation.** Steps 3 and 4 are `CREATE OR REPLACE TABLE ... AS SELECT` with a `GROUP BY`, one table per question.
* **Full refresh.** Every run recomputes each table from scratch. It is simple and safe to re-run, but it reads all of the data every time.

**Why bother with layers**

* *Debugging:* when a number looks wrong, compare layers to find where it changed. Step 5 does exactly that: the bad values are in silver, so they are in gold.
* *Reprocessing:* rebuild silver from bronze whenever the logic changes. You never re-ingest the files.
* *Ownership and access:* engineers own bronze and silver, analysts read gold. Grants are per schema (chapter 11).

**Not here yet:** silver does not validate anything (chapter 06 adds quality rules and quarantine), and nothing is incremental (chapter 08).

## Setup

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

SEVERITIES = ["low", "medium", "high", "critical"]


def show_checks(checks):
    """Show (check, result, expected) rows with a ✅ / ❌ for whether result matches expected."""
    fmt = lambda v: f"{v:,}" if isinstance(v, int) and not isinstance(v, bool) else str(v)
    rows = [(what, fmt(got), fmt(want), "✅" if got == want else "❌") for what, got, want in checks]
    display(spark.createDataFrame(rows, "check string, result string, expected string, ok string"))


needed = ["shield.bronze.incident_reports", "shield.bronze.locations"]
missing = [t for t in needed if not spark.catalog.tableExists(t)]
assert not missing, f"Run notebook 03 first. Missing tables: {missing}"

## 1 · Silver incidents

### Step 1 · Type and deduplicate
**Goal:** one typed row per incident. Bronze keeps replayed deliveries, silver must not.

| Input | Value |
|---|---|
| `shield.bronze.incident_reports` | 10,250 rows (10,000 incidents + 250 replayed duplicates) |

**Expected:** `shield.silver.incident_reports` has 10,000 rows, one per `incident_id`, plus a parsed `reported_ts`.
There is no quality gate yet, so bad values pass through (Step 5).

What you will see:

| check | result | expected | ok |
|---|---|---|---|
| bronze rows | 10,250 | 10,250 | ✅ |
| silver rows | 10,000 | 10,000 | ✅ |
| silver distinct incident_id | 10,000 | 10,000 | ✅ |

`CREATE OR REPLACE` makes the cell safe to re-run, and it also replaces a table that chapter 06 gave a CHECK constraint.

In [ ]:
bronze_incidents = spark.table("shield.bronze.incident_reports")

latest_first = Window.partitionBy("incident_id").orderBy(F.col("_ingested_at").desc())
silver_incidents = (
    bronze_incidents.withColumn("reported_ts", F.expr("try_to_timestamp(reported_at, \"yyyy-MM-dd'T'HH:mm:ss'Z'\")"))
    .withColumn("rn", F.row_number().over(latest_first))
    .filter("rn = 1")
    .drop("rn")
)
silver_incidents.writeTo("shield.silver.incident_reports").createOrReplace()

silver_table = spark.table("shield.silver.incident_reports")
show_checks([
    ("bronze rows", bronze_incidents.count(), 10_250),
    ("silver rows", silver_table.count(), 10_000),
    ("silver distinct incident_id", silver_table.select("incident_id").distinct().count(), 10_000),
])

### Step 2 · Silver locations
**Goal:** a clean lookup table for the joins in the gold steps.

**Expected:** 10 rows, one per `location_id`.

In [ ]:
silver_locations = spark.table("shield.bronze.locations").select("location_id", "city", "country", "lat", "lon")
silver_locations.writeTo("shield.silver.locations").createOrReplace()

show_checks([("silver locations", spark.table("shield.silver.locations").count(), 10)])

## 2 · Gold

### Step 3 · Daily threat briefing
**Goal:** a table named after the question it answers: *how many incidents and casualties per day, city, type and severity?*

**Expected:** incidents without a parsable timestamp are left out (`WHERE reported_ts IS NOT NULL`), so the incidents add up to
10,000 − 97 = 9,903. The table shows the ten most recent days only; the check counts all of them.

In [ ]:
spark.sql(
    """
    CREATE OR REPLACE TABLE shield.gold.daily_threat_briefing AS
    SELECT
      CAST(i.reported_ts AS DATE) AS report_date,
      l.city,
      l.country,
      i.incident_type,
      i.severity,
      COUNT(*)            AS incidents,
      SUM(i.casualties)   AS casualties
    FROM shield.silver.incident_reports i
    JOIN shield.silver.locations l USING (location_id)
    WHERE i.reported_ts IS NOT NULL
    GROUP BY ALL
    """
)
daily_briefing = spark.table("shield.gold.daily_threat_briefing")
display(daily_briefing.orderBy(F.desc("report_date")).limit(10))
show_checks([("incidents in the briefing", daily_briefing.agg(F.sum("incidents")).first()[0], 9_903)])

### Step 4 · Location risk ranking
**Goal:** *which city has the most serious incidents?* Serious means `high` or `critical`.

**Expected:** 10 rows (one per city). This table has no timestamp filter, so its incidents add up to all 10,000.

In [ ]:
spark.sql(
    """
    CREATE OR REPLACE TABLE shield.gold.location_risk AS
    SELECT
      l.city,
      COUNT(*) AS incidents,
      SUM(CASE WHEN i.severity IN ('high','critical') THEN 1 ELSE 0 END) AS serious_incidents,
      ROUND(AVG(i.casualties), 2) AS avg_casualties
    FROM shield.silver.incident_reports i
    JOIN shield.silver.locations l USING (location_id)
    GROUP BY l.city
    """
)
location_risk = spark.table("shield.gold.location_risk")
display(location_risk.orderBy(F.desc("serious_incidents")))
show_checks([
    ("cities", location_risk.count(), 10),
    ("incidents in the ranking", location_risk.agg(F.sum("incidents")).first()[0], 10_000),
])

## 3 · The flaw

### Step 5 · What got through?
**Goal:** prove that silver let bad data pass into the gold numbers.

**Expected:** every defective incident has exactly one defect, 500 in total (5 % of 10,000). The check table counts them in silver:

| check | expected |
|---|---|
| unparsable timestamp | 97 |
| timestamp in the future | 108 |
| severity not in low / medium / high / critical | 109 |
| negative casualties | 91 |
| missing agent | 95 |

The severity breakdown below also lists `catastrophic`, a value that does not exist. Chapter 06 moves these rows to a quarantine table.

In [ ]:
silver_snapshot = spark.table("shield.silver.incident_reports")
defects = silver_snapshot.agg(
    F.sum(F.when(F.col("reported_ts").isNull(), 1).otherwise(0)).alias("unparsable_timestamp"),
    F.sum(F.when(F.col("reported_ts") > F.current_timestamp(), 1).otherwise(0)).alias("future_timestamp"),
    F.sum(F.when(~F.col("severity").isin(SEVERITIES), 1).otherwise(0)).alias("invalid_severity"),
    F.sum(F.when(F.col("casualties") < 0, 1).otherwise(0)).alias("negative_casualties"),
    F.sum(F.when(F.col("agent_id").isNull(), 1).otherwise(0)).alias("missing_agent"),
).first()

show_checks([
    ("unparsable timestamp", defects["unparsable_timestamp"], 97),
    ("timestamp in the future", defects["future_timestamp"], 108),
    ("invalid severity", defects["invalid_severity"], 109),
    ("negative casualties", defects["negative_casualties"], 91),
    ("missing agent", defects["missing_agent"], 95),
])
display(silver_snapshot.groupBy("severity").count().orderBy(F.desc("count")))

## 👀 In the workspace

* **Catalog** → `shield` → `silver` and `gold`: the four tables, with columns and sample data.
* The **Lineage** tab of `gold.location_risk` should show the two silver tables it was built from. Lineage can take a few minutes to appear.

## 🧹 Clean up

Nothing is deleted: chapters 06, 07, 10, 11, 15 and 16 reuse these tables. To start from scratch, run the (commented) cell below.

In [ ]:
# for table in ["gold.location_risk", "gold.daily_threat_briefing", "silver.locations", "silver.incident_reports"]:
#     spark.sql(f"DROP TABLE IF EXISTS shield.{table}")

## 🎯 Recap

* Silver is typed and deduplicated: 10,250 bronze rows became 10,000 incidents. Gold answers one question per table.
* `CREATE OR REPLACE` makes a full refresh safe to re-run, but it recomputes everything every time.
* Without a quality gate, 500 defective incidents reached silver and the gold numbers.

Mini-projects: [docs/05-medallion.md](../../docs/05-medallion.md#try-it-yourself)

> 💡 Next: chapter 06 adds the quality gate and a quarantine table.